# Hückel molecular orbitals of π systems

*chemistry · intermediate* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/chemistry/huckel_mo_theory)

The π electrons of a flat hydrocarbon live in a Hamiltonian that is just the adjacency matrix of its carbon skeleton, $H = \alpha\mathbb 1 + \beta A$. One batched eigh diagonalises seven molecules at once, from ethylene to naphthalene; filling the orbitals gives HOMO–LUMO gaps, delocalisation energies (benzene: $2\beta$, cyclobutadiene: $0$) and Coulson bond orders as an einsum over occupied orbitals, and ring sizes $N = 3, 4, \dots$ show Hückel's $4n + 2$ rule.

**Vector form**

$$H = \alpha\,\mathbb 1 + \beta A,\qquad H\,\mathbf c_k = E_k\,\mathbf c_k,\qquad E_\pi = \sum_k n_k E_k,\qquad P = C\,\mathrm{diag}(n)\,C^{\mathsf T},\qquad E_k^{\text{ring}} = \alpha + 2\beta\cos\frac{2\pi k}{N}$$

**Index (tensor) form**

$$H_{bij} = \alpha\,\delta_{ij} + \beta A_{bij} \qquad H_{bij}C_{bjk} = C_{bik}E_{bk} \qquad E_{\pi,b} = n_{bk}E_{bk} \qquad P_{bij} = n_{bk}\,C_{bik}\,C_{bjk} \qquad q_{bi} = 1 - P_{bii}$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $A_{bij}$ | `tensors.batch(device)` | (B, N, N) | $1$ for bonded carbons $i, j$ of molecule $b$ (atoms one bond length apart), padded to the largest $N = 10$ |
| $H_{bij}$ | `hamiltonian(A, alpha, beta) + pad` | (B, N, N) | Hückel matrix; padding atoms get a huge diagonal so their fake orbitals are never filled |
| $E_{bk},\ C_{bik}$ | `torch.linalg.eigh(H)` | (B, N), (B, N, N) | MO energies (ascending: bonding first, since $\beta < 0$) and coefficients, MO $k$ in column $k$ |
| $n_{bk}$ | `occupations(E, n_electrons)` | (B, N) | Aufbau filling with 2 electrons per MO; a partly filled degenerate set shares its electrons |
| $n_{bk}E_{bk}$ | `einsum('bk,bk->b', n, E)` | (B,) | repeated $k$ ⇒ summed: total π energy $E_\pi$ |
| $E_\pi - E_{\text{loc}}$ | `E_pi - localized_energy(...)` | (B,) | delocalisation energy against isolated double bonds at $2(\alpha + \beta)$ each |
| $n_{bk}C_{bik}C_{bjk}$ | `einsum('bk,bik,bjk->bij', n, C, C)` | (B, N, N) | repeated $k$ ⇒ summed over occupied MOs: Coulson bond order on a bond, π population on the diagonal |
| $q_{bi}$ | `charges(P)` | (B, N) | π charge $1 - P_{bii}$; zero on every atom of these alternant hydrocarbons |
| $E_k^{\text{ring}}$ | `cyclic_levels(N, alpha, beta)` | (N,) | closed-form ring levels, checked against eigh of the cyclic adjacency matrix |

Hückel (1931) kept only what matters for the π electrons of a planar conjugated molecule: one $2p_z$ orbital per carbon, the same energy $\alpha$ on every atom, a hopping energy $\beta$ between bonded neighbours, nothing else and no overlap. The Hamiltonian is then $\alpha\mathbb 1 + \beta A$ with $A$ the graph of the molecule, so its eigenvalues are $\alpha + \beta\lambda$ with $\lambda$ those of $A$; since $\beta < 0$ the most bonding orbital is the one with the largest $\lambda$. Benzene gives $\alpha + 2\beta$, $\alpha + \beta$ (twice), $\alpha - \beta$ (twice), $\alpha - 2\beta$; its six electrons have $E_\pi = 6\alpha + 8\beta$, which is $2\beta$ below three isolated double bonds. Taking $|\beta| \approx 75$ kJ/mol, a rough thermochemical value fitted to benzene's hydrogenation enthalpy (spectroscopy suggests a larger $|\beta|$, so treat it as approximate), this is the famous $\approx 150$ kJ/mol of aromatic stabilisation. Cyclobutadiene has two electrons left over in a degenerate pair at $\alpha$: no delocalisation energy, a zero gap, a diradical, and it is even less stable than open-chain butadiene: antiaromatic. For a ring of $N$ atoms the levels are $\alpha + 2\beta\cos(2\pi k/N)$, one lowest level and then degenerate pairs (Frost's circle), so a closed shell needs $2 + 4n$ electrons: Hückel's rule. Odd rings are shown as their aromatic ions, like the cyclopentadienyl anion and the tropylium cation. The bond orders $P_{ij} = \sum_k n_k C_{ik}C_{jk}$ for butadiene, $0.894$ for the end bonds and $0.447$ for the middle one, match its short-long-short bond lengths; in benzene all six are $2/3$. For alternant hydrocarbons (no odd rings) the pairing theorem makes the spectrum symmetric about $\alpha$ and every π charge zero. Choose another molecule to see its frontier orbitals: the HOMO and LUMO set where electrophiles and nucleophiles attack and, through the gap, the colour (longer polyenes absorb at longer wavelengths).


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/chemistry/huckel_mo_theory/main.py
"""Main loop: carbon skeletons → H = αI + βA → batched eigh → filling, bond orders, 4n + 2 rule → figure."""

from pathlib import Path

import torch

from problems._common.runtime import cli, device

from . import plot, solve, tensors
from .equation import chain_levels, cyclic_levels, is_closed_shell, occupations, pi_energy


def run(params: dict, out_dir: Path) -> Path:
    dev = device()
    alpha, beta = tensors.ALPHA, tensors.BETA  # energies in units of |β| from α
    beta_kj = params["beta"]  # |β| in kJ/mol, only to convert the answers
    names, xys, A, pad, n_el = tensors.batch(dev)
    res = solve.solve(A, pad, n_el, alpha, beta)
    print(f"{len(names)} molecules in one batch, H of shape {tuple(A.shape)}; energies E = α + xβ, β < 0")
    print(f"{'molecule':15s} {'x of each MO (bonding first)':44s} {'gap/|β|':>8s} {'deloc./|β|':>10s} {'kJ/mol':>7s}")
    for b, name in enumerate(names):
        N = len(xys[b])
        x = ((res["E"][b, :N] - alpha) / beta).round(decimals=6) + 0.0  # E = α + xβ (+ 0.0 drops "-0.000")
        de = round((res["deloc"][b] / beta).item(), 6) + 0.0  # in units of β: positive = stabilised (β < 0)
        gap = round(res["gap"][b].item(), 6) + 0.0
        xs = " ".join(f"{v:+.3f}" for v in x.tolist())
        print(f"{name:15s} {xs:44s} {gap:8.3f} {de:10.3f} {de * beta_kj:7.0f}")

    b = names.index("butadiene")
    P = res["P"][b]
    print(f"butadiene π bond orders: C1–C2 = {P[0, 1]:.3f}, C2–C3 = {P[1, 2]:.3f}  (Coulson: 0.894, 0.447)")
    b = names.index("benzene")
    print(f"benzene π bond order {res['P'][b, 0, 1]:.3f} (2/3), π charges max |q| = {res['q'][b, :6].abs().max():.1e}")

    # Hückel's 4n + 2 rule: eigh of every ring against the closed form α + 2β cos(2πk/N)
    rings = []
    for N in range(3, params["ring_max"] + 1):
        E_num = torch.linalg.eigvalsh(alpha * torch.eye(N, dtype=torch.float64) + beta * tensors.ring_adjacency(N))
        E = cyclic_levels(N, alpha, beta)
        ne = tensors.ring_electrons(N)
        closed = is_closed_shell(E, ne)
        n = occupations(E[None], [ne])[0]
        chain = chain_levels(N, alpha, beta)
        e_ring = pi_energy(E[None], n[None])[0]
        e_chain = pi_energy(chain[None], occupations(chain[None], [ne]))[0]
        rings.append({"N": N, "ne": ne, "E": E, "n": n, "closed": closed, "ring_minus_chain": e_ring - e_chain})
        kind = "closed shell, aromatic" if closed else "half-filled degenerate pair, antiaromatic"
        print(
            f"ring N = {N:2d}, {ne:2d} π e⁻: max |eigh - closed form| = {(E_num - E).abs().max():.1e}, {kind}, "
            f"E_ring - E_chain = {(e_ring - e_chain).item():+.3f}|β|"
        )
    return plot.figure(names, xys, res, rings, params["molecule"], beta_kj, out_dir / "huckel_mo_theory.png")


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/chemistry/huckel_mo_theory/equation.py
r"""Physics → tensor translation: Hückel molecular-orbital theory for π electrons.

Indices: b molecule, i and j carbon atoms, k molecular orbital (MO).

Each carbon contributes one 2p orbital perpendicular to the molecular plane. In that basis the π Hamiltonian is
    H_bij = α δ_ij + β A_bij,
α the energy of an electron in an isolated 2p orbital, β < 0 the resonance (hopping) integral between bonded
neighbours and A the adjacency matrix of the carbon skeleton. Diagonalising, H C = C diag(E), gives the MO
energies E_bk and the coefficients C_bik of atom i in MO k.

With n_bk electrons in MO k (0, 1 or 2):
    E_π       = n_bk E_bk                         (k summed)
    P_bij     = n_bk C_bik C_bjk                  (k summed): Coulson bond order on a bond, π population on i = j
    q_bi      = 1 - P_bii                         π charge of atom i in a neutral molecule
For a ring of N atoms the eigenvalues are known in closed form, E_k = α + 2β cos(2πk / N), k = 0 … N - 1.
"""

import math

import torch


def hamiltonian(A: torch.Tensor, alpha: float, beta: float) -> torch.Tensor:
    """H_bij = α δ_ij + β A_bij.   A (B, N, N) → (B, N, N)"""
    eye = torch.eye(A.shape[-1], dtype=A.dtype, device=A.device)
    return alpha * eye + beta * A


def occupations(E: torch.Tensor, n_electrons: list[int], tol: float = 1e-6) -> torch.Tensor:
    """Aufbau filling n_bk: two electrons per MO from the lowest energy up. A partly filled set of degenerate MOs
    shares its electrons equally (Hund-like averaging), so the result does not depend on how eigh rotates them.

    E (B, N) ascending, n_electrons[b] → (B, N)"""
    n = torch.zeros_like(E)
    for b, left in enumerate(n_electrons):
        k = 0
        while left > 0:
            shell = (E[b] - E[b, k]).abs() < tol  # this level and every MO degenerate with it
            g = int(shell.sum())
            take = min(left, 2 * g)
            n[b, shell] = take / g
            left -= take
            k += g
    return n


def pi_energy(E: torch.Tensor, n: torch.Tensor) -> torch.Tensor:
    """E_π = n_bk E_bk, k summed.   (B,)"""
    return torch.einsum("bk,bk->b", n, E)


def localized_energy(n_electrons: list[int], alpha: float, beta: float) -> torch.Tensor:
    """Energy of isolated ethylene-like double bonds (2 electrons at α + β each) plus a lone electron at α
    if the count is odd: the reference for the delocalisation energy.   (B,)"""
    return torch.tensor([(ne // 2) * 2 * (alpha + beta) + (ne % 2) * alpha for ne in n_electrons], dtype=torch.float64)


def bond_order_matrix(C: torch.Tensor, n: torch.Tensor) -> torch.Tensor:
    """P_bij = n_bk C_bik C_bjk, k summed over the occupied MOs.   (B, N, N)"""
    return torch.einsum("bk,bik,bjk->bij", n, C, C)


def charges(P: torch.Tensor) -> torch.Tensor:
    """q_bi = 1 - P_bii: zero for every atom of an alternant hydrocarbon (Coulson–Rushbrooke).   (B, N)"""
    return 1 - torch.diagonal(P, dim1=-2, dim2=-1)


def homo_lumo_gap(E: torch.Tensor, n: torch.Tensor) -> torch.Tensor:
    """LUMO minus HOMO energy: lowest MO with room for an electron minus highest MO holding one.   (B,)"""
    big = torch.finfo(E.dtype).max
    homo = torch.where(n > 0, E, -big).max(-1).values
    lumo = torch.where(n < 2, E, big).min(-1).values
    return lumo - homo


def cyclic_levels(N: int, alpha: float, beta: float) -> torch.Tensor:
    """Closed-form eigenvalues of a ring of N atoms, E_k = α + 2β cos(2πk/N), sorted ascending.   (N,)"""
    k = torch.arange(N, dtype=torch.float64)
    return (alpha + 2 * beta * torch.cos(2 * math.pi * k / N)).sort().values


def chain_levels(N: int, alpha: float, beta: float) -> torch.Tensor:
    """Closed-form eigenvalues of an open chain of N atoms, E_k = α + 2β cos(πk/(N+1)), k = 1 … N.   (N,)"""
    k = torch.arange(1, N + 1, dtype=torch.float64)
    return (alpha + 2 * beta * torch.cos(math.pi * k / (N + 1))).sort().values


def is_closed_shell(levels: torch.Tensor, n_electrons: int, tol: float = 1e-6) -> bool:
    """True when the electrons exactly fill a set of degenerate levels: a gap separates HOMO and LUMO.
    For a ring this happens for 4n + 2 electrons (Hückel's rule); 4n leaves a degenerate pair half filled."""
    n = occupations(levels[None], [n_electrons], tol)[0]
    return bool(homo_lumo_gap(levels[None], n[None])[0] > tol)


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/chemistry/huckel_mo_theory/tensors.py
"""Build the concrete tensors: carbon skeletons, their adjacency matrices, padded into one batch.

Energies are in units of |β| measured from α: ALPHA = 0 and BETA = -1, so an MO at α + xβ has energy -x.
"""

import math

import torch

ALPHA = 0.0
BETA = -1.0
PAD = 1e3  # diagonal entry for padding atoms: their fake MOs sit far above every real one and are never filled


def _zigzag(n: int) -> list[tuple[float, float]]:
    """An all-trans chain with 120° angles and unit bonds."""
    return [(i * math.sqrt(3) / 2, 0.5 * (i % 2)) for i in range(n)]


def _ring(n: int, radius: float, start: float = 90.0) -> list[tuple[float, float]]:
    return [
        (radius * math.cos(math.radians(start + 360 * i / n)), radius * math.sin(math.radians(start + 360 * i / n)))
        for i in range(n)
    ]


def _naphthalene() -> list[tuple[float, float]]:
    s = math.sqrt(3) / 2
    left = [(-s, 1), (-2 * s, 0.5), (-2 * s, -0.5), (-s, -1)]
    right = [(s, 1), (2 * s, 0.5), (2 * s, -0.5), (s, -1)]
    return [(0, 0.5), (0, -0.5)] + left + right  # the two shared atoms first


# name → (atom positions with unit C–C bonds, number of π electrons); the allyl radical has an odd count
MOLECULES = {
    "ethylene": (_zigzag(2), 2),
    "allyl": (_zigzag(3), 3),
    "butadiene": (_zigzag(4), 4),
    "hexatriene": (_zigzag(6), 6),
    "cyclobutadiene": ([(-0.5, -0.5), (0.5, -0.5), (0.5, 0.5), (-0.5, 0.5)], 4),
    "benzene": (_ring(6, 1.0), 6),
    "naphthalene": (_naphthalene(), 10),
}


def adjacency(xy: torch.Tensor) -> torch.Tensor:
    """A_ij = 1 for atoms one bond length (1) apart, else 0.   xy (N, 2) → (N, N)"""
    d = torch.cdist(xy, xy)
    return ((d > 0.9) & (d < 1.1)).to(xy.dtype)


def batch(device: torch.device) -> tuple[list[str], list[torch.Tensor], torch.Tensor, torch.Tensor, list[int]]:
    """Every molecule's adjacency matrix in one padded batch.

    Returns names, positions [(N_b, 2)], A (B, N, N), a padding diagonal D (B, N, N) and electron counts.
    Molecule b fills the top-left N_b × N_b block; its padding atoms get D_ii = PAD and no bonds.
    """
    names = list(MOLECULES)
    xys = [torch.tensor(MOLECULES[m][0], dtype=torch.float64) for m in names]
    N = max(len(xy) for xy in xys)
    A = torch.zeros(len(names), N, N, dtype=torch.float64)
    D = torch.zeros_like(A)
    for b, xy in enumerate(xys):
        n = len(xy)
        A[b, :n, :n] = adjacency(xy)
        D[b, range(n, N), range(n, N)] = PAD
    return names, xys, A.to(device), D.to(device), [MOLECULES[m][1] for m in names]


def ring_adjacency(N: int) -> torch.Tensor:
    """A cycle of N atoms: A_ij = 1 when j = i ± 1 (mod N).   (N, N)"""
    eye = torch.eye(N, dtype=torch.float64)
    return eye.roll(1, 1) + eye.roll(-1, 1)


def ring_electrons(N: int) -> int:
    """π electrons of the ring studied for each size: the neutral molecule for even N; for odd N the ion
    (cation or anion) with 4n + 2 electrons, like C₃H₃⁺, C₅H₅⁻ and C₇H₇⁺."""
    if N % 2 == 0:
        return N
    return N - 1 if (N - 1) % 4 == 2 else N + 1


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/chemistry/huckel_mo_theory/solve.py
"""The solver: one batched eigh for all molecules, then fill the MOs and form bond orders and charges."""

import torch

from .equation import (
    bond_order_matrix,
    charges,
    hamiltonian,
    homo_lumo_gap,
    localized_energy,
    occupations,
    pi_energy,
)


def solve(A: torch.Tensor, pad: torch.Tensor, n_electrons: list[int], alpha: float, beta: float) -> dict:
    """Diagonalise H_bij = α δ_ij + β A_bij (+ padding) for the whole batch at once.

    Returns a dict of E (B, N) ascending, C (B, N, N) with MO k in column k, occupations n (B, N),
    E_pi, E_loc, delocalisation energy, HOMO–LUMO gap (B,), bond orders P (B, N, N) and charges q (B, N).
    """
    H = hamiltonian(A, alpha, beta) + pad  # (B, N, N)
    E, C = torch.linalg.eigh(H)  # (B, N), (B, N, N)
    n = occupations(E, n_electrons)
    E_pi = pi_energy(E, n)
    E_loc = localized_energy(n_electrons, alpha, beta).to(E.device)
    P = bond_order_matrix(C, n)
    return {
        "E": E,
        "C": C,
        "n": n,
        "E_pi": E_pi,
        "E_loc": E_loc,
        "deloc": E_pi - E_loc,
        "gap": homo_lumo_gap(E, n),
        "P": P,
        "q": charges(P),
    }


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/chemistry/huckel_mo_theory/plot.py
"""Static figure: MO energy diagrams with electron filling, the 4n + 2 rule across ring sizes, delocalisation
energies and gaps, and the MO lobes and bond orders of one molecule drawn on its skeleton."""

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from problems._common.plotting import save_png

BLUE, RED, GREY, AMBER = "#2563eb", "#dc2626", "#9ca3af", "#f59e0b"


def _shells(E: np.ndarray, tol: float = 1e-6) -> list[list[int]]:
    """Group the indices of degenerate levels."""
    groups: list[list[int]] = []
    for k, e in enumerate(E):
        if groups and abs(E[groups[-1][0]] - e) < tol:
            groups[-1].append(k)
        else:
            groups.append([k])
    return groups


def _levels(ax, x0: float, E: np.ndarray, n: np.ndarray, width: float = 0.7, colour: str = "#374151"):
    """Draw MO levels at column x0 (degenerate ones side by side) with ↑↓ electrons."""
    for shell in _shells(E):
        g = len(shell)
        w = width / g
        for s, k in enumerate(shell):
            xc = x0 - width / 2 + w * (s + 0.5)
            ax.plot([xc - 0.42 * w, xc + 0.42 * w], [E[k], E[k]], color=colour, lw=2, solid_capstyle="butt")
            if n[k] >= 1:
                ax.plot(xc - 0.1 * w if n[k] > 1.5 else xc, E[k], "^", color=BLUE, ms=4)
            if n[k] > 1.5:
                ax.plot(xc + 0.1 * w, E[k], "v", color=BLUE, ms=4)
            if 0 < n[k] < 1:
                ax.plot(xc, E[k], "^", mfc="none", mec=BLUE, ms=4)


def _skeleton(ax, xy: np.ndarray, A: np.ndarray, lw: float = 1.5):
    i, j = np.nonzero(np.triu(A))
    for a, b in zip(i, j):
        ax.plot(xy[[a, b], 0], xy[[a, b], 1], color=GREY, lw=lw, zorder=1)
    ax.set_aspect("equal")
    ax.axis("off")


def figure(names, xys, res, rings, molecule, beta_kj, path: Path) -> Path:
    E = res["E"].cpu().numpy()
    n = res["n"].cpu().numpy()
    C = res["C"].cpu().numpy()
    P = res["P"].cpu().numpy()
    gap = res["gap"].cpu().numpy().round(9) + 0.0  # + 0.0 turns -0.0 into 0.0
    deloc = -res["deloc"].cpu().numpy().round(9) + 0.0  # in units of |β|, positive = stabilised
    B = len(names)

    fig = plt.figure(figsize=(11, 9))
    outer = fig.add_gridspec(
        3, 1, height_ratios=[1.15, 1.0, 1.0], hspace=0.42, left=0.06, right=0.98, top=0.95, bottom=0.04
    )
    ax_mo = fig.add_subplot(outer[0])
    mid = outer[1].subgridspec(1, 2, width_ratios=[1.6, 1], wspace=0.22)
    ax_ring, ax_bar = fig.add_subplot(mid[0]), fig.add_subplot(mid[1])
    low = outer[2].subgridspec(1, 5, wspace=0.05)
    ax_lobes = [fig.add_subplot(low[c]) for c in range(5)]

    # --- MO energy diagrams of the whole batch ---
    for b in range(B):
        N = len(xys[b])
        _levels(ax_mo, b, E[b, :N], n[b, :N])
        occ = np.nonzero(n[b, :N] > 0)[0]
        homo, lumo = E[b, occ[-1]], E[b, :N][n[b, :N] < 2].min()
        if gap[b] > 1e-6:
            ax_mo.annotate("", (b + 0.45, lumo), (b + 0.45, homo), arrowprops={"arrowstyle": "<->", "color": AMBER})
    ax_mo.axhline(0, color=GREY, lw=0.8, ls=(0, (4, 3)))
    ax_mo.text(B - 0.45, 0.06, "α", color="#6b7280", fontsize=9)
    ax_mo.set_xticks(range(B), [f"{m}\ngap {g:.2f}|β|" for m, g in zip(names, gap)], fontsize=8)
    ax_mo.set_xlim(-0.6, B - 0.3)
    ax_mo.set_ylim(-2.6, 2.6)  # bonding (E < α, since β < 0) at the bottom
    ax_mo.set_ylabel("E - α  (units of |β|)")
    ax_mo.set_yticks([-2, -1, 0, 1, 2], ["α + 2β", "α + β", "α", "α - β", "α - 2β"])
    ax_mo.set_title("Hückel MO levels from one batched eigh of H = αI + βA  (↑↓ electrons, amber: HOMO–LUMO gap)")
    ax_mo.grid(axis="x", visible=False)

    # --- 4n + 2 rule: closed-form ring levels with their filling ---
    for r in rings:
        colour = BLUE if r["closed"] else RED
        _levels(ax_ring, r["N"], r["E"].numpy(), r["n"].numpy(), width=0.8, colour=colour)
    ax_ring.axhline(0, color=GREY, lw=0.8, ls=(0, (4, 3)))
    Ns = [r["N"] for r in rings]
    ax_ring.set_xticks(Ns, [f"N={r['N']}\n{r['ne']} e⁻" for r in rings], fontsize=7)
    ax_ring.set_xlim(Ns[0] - 0.6, Ns[-1] + 0.6)
    ax_ring.set_ylim(-2.4, 2.4)
    ax_ring.set_yticks([-2, -1, 0, 1, 2], ["α+2β", "α+β", "α", "α-β", "α-2β"], fontsize=8)
    ax_ring.set_title(
        "rings: E_k = α + 2β cos(2πk/N);  blue: 4n+2 e⁻ closed shell (aromatic), red: 4n (antiaromatic)", fontsize=9
    )
    ax_ring.grid(axis="x", visible=False)

    # --- delocalisation energies and gaps ---
    xb = np.arange(B)
    ax_bar.bar(xb - 0.2, deloc, 0.4, color=BLUE, label="delocalisation energy")
    ax_bar.bar(xb + 0.2, gap, 0.4, color=AMBER, label="HOMO–LUMO gap")
    ax_bar.set_xticks(xb, [m[:6] for m in names], fontsize=7, rotation=30)
    ax_bar.set_ylabel("units of |β|")
    ax_bar.legend(fontsize=7, loc="upper left", frameon=False)
    ib, ic = names.index("benzene"), names.index("cyclobutadiene")
    rc = {r["N"]: r["ring_minus_chain"].item() for r in rings}
    ax_bar.set_ylim(0, 1.6 * max(deloc.max(), gap.max()))
    ax_bar.text(
        0.02,
        0.80,
        f"benzene {deloc[ib]:.2f}|β| ≈ {deloc[ib] * beta_kj:.0f} kJ/mol\n"
        f"cyclobutadiene {deloc[ic]:.2f}|β|\n"
        f"ring − chain: C₆ {rc.get(6, np.nan):+.2f}|β|, C₄ {rc.get(4, np.nan):+.2f}|β|",
        transform=ax_bar.transAxes,
        ha="left",
        va="top",
        fontsize=7,
    )
    ax_bar.set_title("delocalisation energy and gap", fontsize=9)

    # --- MO lobes on the skeleton of the chosen molecule, plus its bond orders ---
    b = names.index(molecule)
    xy = xys[b].numpy()
    N = len(xy)
    d = np.linalg.norm(xy[:, None] - xy[None], axis=-1)
    A = ((d > 0.9) & (d < 1.1)).astype(float)
    homo = int(np.nonzero(n[b, :N] > 0)[0][-1])
    ks = list(range(N)) if N <= 4 else [homo - 1, homo, homo + 1, homo + 2]
    span = np.ptp(xy, 0).max() + 1.3
    centre = xy.mean(0)
    for c, ax in enumerate(ax_lobes[:4]):
        if c >= len(ks):
            ax.axis("off")
            continue
        k = ks[c]
        _skeleton(ax, xy, A)
        for i in range(N):
            ci = C[b, i, k]
            if abs(ci) > 1e-3:
                ax.add_patch(plt.Circle(xy[i], 0.75 * abs(ci), color=BLUE if ci > 0 else RED, alpha=0.75, zorder=2))
        x = -E[b, k]
        tag = "HOMO" if k == homo else ("LUMO" if k == homo + 1 else "")
        ax.set_title(f"ψ{k + 1} {tag}  E = α {'+' if x >= 0 else '−'} {abs(x):.3f}β", fontsize=8)
        ax.set_xlim(xy[:, 0].mean() - span / 2, xy[:, 0].mean() + span / 2)
        ax.set_ylim(xy[:, 1].mean() - span / 2, xy[:, 1].mean() + span / 2)
    ax = ax_lobes[4]
    _skeleton(ax, xy, A, lw=0)
    i, j = np.nonzero(np.triu(A))
    for a, c2 in zip(i, j):
        p = P[b, a, c2]
        ax.plot(xy[[a, c2], 0], xy[[a, c2], 1], color="#374151", lw=1 + 4 * p, zorder=1)
        mid = xy[[a, c2]].mean(0)
        normal = np.array([xy[a, 1] - xy[c2, 1], xy[c2, 0] - xy[a, 0]])  # perpendicular to the bond, unit length
        if normal @ (mid - centre) < 0 or np.allclose(mid, centre):
            normal = -normal  # point away from the middle of the molecule
        ax.text(*(mid + 0.35 * normal), f"{p:.3f}", fontsize=7, ha="center", va="center", color=BLUE)
    ax.plot(xy[:, 0], xy[:, 1], "o", color="#374151", ms=4)
    ax.set_title(f"{molecule}: π bond orders P_ij", fontsize=8)
    ax.set_xlim(xy[:, 0].mean() - span / 2, xy[:, 0].mean() + span / 2)
    ax.set_ylim(xy[:, 1].mean() - span / 2, xy[:, 1].mean() + span / 2)
    fig.text(0.5, 0.015, f"MO lobes of {molecule}: radius ∝ |C_ik|, blue +, red −", fontsize=9, ha="center")
    return save_png(fig, path)


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'molecule': 'butadiene',
    'ring_max': 10,
    'beta': 75.0
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.chemistry.huckel_mo_theory")]:
    del sys.modules[name]
main = importlib.import_module("problems.chemistry.huckel_mo_theory.main")
problem_dir = Path("problems/chemistry/huckel_mo_theory")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
